# LeetCode 2091 — Removing Minimum and Maximum From Array bằng Python

Link đề: https://leetcode.com/problems/removing-minimum-and-maximum-from-array/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimumDeletions(self, nums: List[int]) -> int:
        ...
```

Lộ trình:
1. **Brute force** thử mọi cặp (số lần xóa đầu `a`, số lần xóa cuối `b`) — `O(n²)`.
2. **Duyệt `a`, tính `b` nhỏ nhất cần thiết** — `O(n)`.
3. **Ba trường hợp** (cả hai từ đầu / cả hai từ cuối / mỗi bên một) — `O(n)` để tìm chỉ số, `O(1)` để tính.
4. **BFS trên trạng thái `(l, r)`** — mô hình hóa tổng quát, dùng để kiểm chứng độc lập.

## 1. Mục tiêu học

1. Nhận ra rằng sau mọi thao tác, phần còn lại luôn là một **đoạn liên tiếp** `nums[a .. n − 1 − b]` — bài trở thành chọn `a`, `b`.
2. Rút gọn về hai chỉ số `i < j` (vị trí của min và max, sắp theo thứ tự) và **ba phương án** đóng.
3. Đếm đúng số lần xóa: xóa phần tử ở chỉ số `t` từ đầu cần `t + 1` lần, từ cuối cần `n − t` lần.
4. Thấy vì sao **tham lam từng bước** ("xóa phía gần mục tiêu tiếp theo") có thể sai, và kiểm chứng bằng BFS.
5. Mở rộng sang "xóa một tập chỉ số bất kỳ bằng thao tác ở hai đầu".

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm các số nguyên **phân biệt**. Phần tử nhỏ nhất (min) và lớn nhất (max) cần bị xóa. Mỗi thao tác xóa **phần tử đầu** hoặc **phần tử cuối** của mảng. Trả về số thao tác ít nhất để cả min và max đều đã bị xóa.

**Ví dụ 1:** `[2,10,7,5,4,1,8,6]` → `5` (min = 1 ở chỉ số 5, max = 10 ở chỉ số 1; xóa 2 phần tử đầu + 3 phần tử cuối).

**Ví dụ 2:** `[0,-4,19,1,8,-2,-3,5]` → `3` (min ở 1, max ở 2; xóa 3 phần tử đầu).

**Ví dụ 3:** `[101]` → `1` (phần tử duy nhất vừa là min vừa là max).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `-10⁵ ≤ nums[i] ≤ 10⁵`, các số phân biệt.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [([2, 10, 7, 5, 4, 1, 8, 6], 5), ([0, -4, 19, 1, 8, -2, -3, 5], 3), ([101], 1)]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[2, 10, 7, 5, 4, 1, 8, 6] -> kỳ vọng 5
[0, -4, 19, 1, 8, -2, -3, 5] -> kỳ vọng 3
[101] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Tìm chỉ số của min và max trong một lượt

`nums.index(min(nums))` duyệt hai lần; một lượt thủ công cũng được. Cả hai `O(n)`.

In [2]:
nums = [2, 10, 7, 5, 4, 1, 8, 6]
print("index min:", nums.index(min(nums)), "| index max:", nums.index(max(nums)))
imin = imax = 0
for t, x in enumerate(nums):
    if x < nums[imin]: imin = t
    if x > nums[imax]: imax = t
print("một lượt:", imin, imax)

index min: 5 | index max: 1
một lượt: 5 1


### 3.2 Số thao tác để xóa phần tử ở chỉ số `t`

- Từ đầu: phải xóa `nums[0..t]` ⇒ `t + 1` thao tác.
- Từ cuối: phải xóa `nums[t..n−1]` ⇒ `n − t` thao tác.

In [3]:
n = len(nums)
for t in (0, 3, n - 1):
    print(f"t = {t}: từ đầu {t + 1}, từ cuối {n - t}")

t = 0: từ đầu 1, từ cuối 8
t = 3: từ đầu 4, từ cuối 5
t = 7: từ đầu 8, từ cuối 1


### 3.3 `deque` để mô phỏng xóa hai đầu

`popleft()` và `pop()` đều `O(1)` — dùng cho mô phỏng/BFS.

In [4]:
d = deque([2, 10, 7, 5])
d.popleft(); d.pop()
print(d)

deque([10, 7])


## 4. Ý tưởng nền tảng / trực giác

Gọi `i < j` là hai chỉ số của min và max (sắp lại theo vị trí). Có đúng ba cách "hợp lý":

```
chỉ số:  0 ........ i ........ j ........ n-1
(A) xóa từ đầu tới j:          [=========== j]                -> j + 1
(B) xóa từ cuối tới i:              [i ===========]           -> n - i
(C) đầu tới i, cuối tới j:     [== i]            [j ==]       -> (i + 1) + (n - j)
```

Ví dụ 1: `n = 8`, `i = 1` (max), `j = 5` (min).

| phương án | công thức | giá trị |
|---|---|---|
| (A) cả hai từ đầu | `j + 1` | 6 |
| (B) cả hai từ cuối | `n − i` | 7 |
| (C) mỗi bên một | `(i + 1) + (n − j)` | 2 + 3 = **5** |

Ví dụ 2: `i = 1`, `j = 2`, `n = 8` ⇒ `(A) 3`, `(B) 7`, `(C) 2 + 6 = 8` ⇒ **3**.

In [5]:
def three_options(nums):
    n = len(nums)
    i, j = sorted((nums.index(min(nums)), nums.index(max(nums))))
    return {"A (đầu)": j + 1, "B (cuối)": n - i, "C (hai bên)": (i + 1) + (n - j)}

for nums, want in examples:
    opts = three_options(nums)
    print(nums, opts, "-> min =", min(opts.values()), "| kỳ vọng", want)

[2, 10, 7, 5, 4, 1, 8, 6] {'A (đầu)': 6, 'B (cuối)': 7, 'C (hai bên)': 5} -> min = 5 | kỳ vọng 5
[0, -4, 19, 1, 8, -2, -3, 5] {'A (đầu)': 3, 'B (cuối)': 7, 'C (hai bên)': 8} -> min = 3 | kỳ vọng 3
[101] {'A (đầu)': 1, 'B (cuối)': 1, 'C (hai bên)': 2} -> min = 1 | kỳ vọng 1


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp `(a, b)`

**Ý tưởng:** xóa `a` phần tử đầu và `b` phần tử cuối (`a + b ≤ n`) ⇒ phần còn lại là `nums[a .. n−1−b]`. Hợp lệ nếu cả hai chỉ số min/max nằm **ngoài** đoạn còn lại. Thứ tự các thao tác không quan trọng — chỉ số lượng mỗi đầu.

In [6]:
def deletions_brute(nums):
    n = len(nums)
    imin, imax = nums.index(min(nums)), nums.index(max(nums))
    removed = lambda t, a, b: t < a or t >= n - b
    best = n
    for a in range(n + 1):
        for b in range(n - a + 1):
            if removed(imin, a, b) and removed(imax, a, b):
                best = min(best, a + b)
    return best

for nums, want in examples:
    print(deletions_brute(nums), "kỳ vọng", want)

5 kỳ vọng 5
3 kỳ vọng 3
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` cặp ⇒ `5·10⁹` với `n = 10⁵` — quá chậm. Space `O(1)`.

**Nút thắt:** với `a` cố định, `b` nhỏ nhất cần thiết xác định **trực tiếp** — không cần thử mọi `b`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (`b` tối thiểu khi biết `a`):** các chỉ số mục tiêu chưa bị xóa từ đầu (tức `≥ a`) phải bị xóa từ cuối; `b` nhỏ nhất `= n − (chỉ số mục tiêu nhỏ nhất còn ≥ a)`, hoặc `0` nếu không còn mục tiêu nào.

**Insight 2 (chỉ cần thử ba `a`):** chi phí `a + b(a)` là hàm bậc thang: trong khoảng giữa hai mục tiêu, tăng `a` chỉ làm tổng tăng mà `b` không giảm. Nên chỉ đáng thử `a ∈ {0, i + 1, j + 1}` ⇔ ba phương án (B), (C), (A).

**Quy tắc:** với `i ≤ j`:

`ans = min(j + 1, n − i, (i + 1) + (n − j))`.

Khi `n = 1` thì `i = j = 0` ⇒ `min(1, 1, 2) = 1` ✓ — công thức tự xử lý trường hợp min trùng max.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt `a`, tính `b` tối thiểu

In [7]:
def deletions_scan_a(nums):
    n = len(nums)
    targets = sorted({nums.index(min(nums)), nums.index(max(nums))})
    best = n
    for a in range(n + 1):
        rest = [t for t in targets if t >= a]          # tối đa 2 phần tử
        b = n - rest[0] if rest else 0
        if a + b <= n:
            best = min(best, a + b)
    return best

for nums, want in examples:
    print(deletions_scan_a(nums), "kỳ vọng", want)

5 kỳ vọng 5
3 kỳ vọng 3
1 kỳ vọng 1


**Độ phức tạp:** `O(n)` thời gian (mỗi `a` xử lý tối đa 2 mục tiêu), `O(1)` bộ nhớ. Cách này dễ tổng quát hóa cho **nhiều** mục tiêu (Bài 5).

### 7.2 Cách 3 — Ba trường hợp (code nộp)

In [8]:
def deletions_three_cases(nums):
    n = len(nums)
    i = j = 0
    for t, x in enumerate(nums):
        if x < nums[i]: i = t
        if x > nums[j]: j = t
    if i > j:
        i, j = j, i
    return min(j + 1, n - i, i + 1 + n - j)

for nums, want in examples:
    print(deletions_three_cases(nums), "kỳ vọng", want)

5 kỳ vọng 5
3 kỳ vọng 3
1 kỳ vọng 1


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n)` để tìm chỉ số, `O(1)` để tính. Space `O(1)`.

### 7.3 Cách 4 — BFS trên trạng thái `(l, r)` (mô hình hóa tổng quát)

Trạng thái = đoạn còn lại `[l, r]`. Mỗi thao tác: `l += 1` hoặc `r -= 1`. Đích: cả hai mục tiêu nằm ngoài `[l, r]`. BFS tìm số bước ít nhất. Không cần cho bài này, nhưng là "trọng tài độc lập" (không dùng nhận xét `(a, b)`), và giúp thấy rõ không gian lựa chọn.

In [9]:
def deletions_bfs(nums):
    n = len(nums)
    tg = {nums.index(min(nums)), nums.index(max(nums))}
    done = lambda l, r: all(t < l or t > r for t in tg)
    start = (0, n - 1)
    q = deque([(start, 0)]); seen = {start}
    while q:
        (l, r), d = q.popleft()
        if done(l, r):
            return d
        for nxt in ((l + 1, r), (l, r - 1)):
            if nxt not in seen:
                seen.add(nxt); q.append((nxt, d + 1))

for nums, want in examples:
    print(deletions_bfs(nums), "kỳ vọng", want)

5 kỳ vọng 5
3 kỳ vọng 3
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` trạng thái — chỉ dùng với `n` nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên ba ví dụ: chỉ số min/max, sau khi sắp, ba phương án và lựa chọn.

In [10]:
def trace_three(nums):
    n = len(nums); i = j = 0
    for t, x in enumerate(nums):
        if x < nums[i]: i = t
        if x > nums[j]: j = t
    print(f"nums = {nums}")
    print(f"  min = {nums[i]} tại {i}, max = {nums[j]} tại {j}")
    if i > j: i, j = j, i
    a, b, c = j + 1, n - i, i + 1 + n - j
    best = min(a, b, c)
    tag = {a: "A: xóa từ đầu", b: "B: xóa từ cuối", c: "C: hai bên"}[best]
    print(f"  i = {i}, j = {j}, n = {n}: A = {a}, B = {b}, C = {i + 1} + {n - j} = {c} -> {best} ({tag})")
    return best

for nums, _ in examples:
    trace_three(nums)

nums = [2, 10, 7, 5, 4, 1, 8, 6]
  min = 1 tại 5, max = 10 tại 1
  i = 1, j = 5, n = 8: A = 6, B = 7, C = 2 + 3 = 5 -> 5 (C: hai bên)
nums = [0, -4, 19, 1, 8, -2, -3, 5]
  min = -4 tại 1, max = 19 tại 2
  i = 1, j = 2, n = 8: A = 3, B = 7, C = 2 + 6 = 8 -> 3 (A: xóa từ đầu)
nums = [101]
  min = 101 tại 0, max = 101 tại 0
  i = 0, j = 0, n = 1: A = 1, B = 1, C = 1 + 1 = 2 -> 1 (B: xóa từ cuối)


In [11]:
def trace_scan(nums):
    n = len(nums); targets = sorted({nums.index(min(nums)), nums.index(max(nums))}); best = n
    print(f"mục tiêu (chỉ số): {targets}")
    print(f"{'a':>2} | mục tiêu còn lại | b cần | a + b | best")
    for a in range(n + 1):
        rest = [t for t in targets if t >= a]; b = n - rest[0] if rest else 0
        if a + b <= n: best = min(best, a + b)
        print(f"{a:>2} | {str(rest):>16} | {b:>5} | {a + b:>5} | {best}")

trace_scan([2, 10, 7, 5, 4, 1, 8, 6])

mục tiêu (chỉ số): [1, 5]
 a | mục tiêu còn lại | b cần | a + b | best
 0 |           [1, 5] |     7 |     7 | 7
 1 |           [1, 5] |     7 |     8 | 7
 2 |              [5] |     3 |     5 | 5
 3 |              [5] |     3 |     6 | 5
 4 |              [5] |     3 |     7 | 5
 5 |              [5] |     3 |     8 | 5
 6 |               [] |     0 |     6 | 5
 7 |               [] |     0 |     7 | 5
 8 |               [] |     0 |     8 | 5


Đọc bảng: tổng `a + b` chỉ "giảm đột ngột" ngay **sau** khi `a` vượt qua một mục tiêu (`a = 2` và `a = 6`), còn giữa các mục tiêu thì tăng dần — đúng Insight 2: chỉ cần thử `a ∈ {0, i + 1, j + 1}`.

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minimumDeletions(self, nums: List[int]) -> int:
        n = len(nums)
        i = j = 0
        for t, x in enumerate(nums):
            if x < nums[i]:
                i = t
            if x > nums[j]:
                j = t
        if i > j:
            i, j = j, i
        return min(j + 1,            # cả hai từ đầu
                   n - i,            # cả hai từ cuối
                   i + 1 + n - j)    # mỗi bên một

sol = Solution()
for nums, want in examples:
    got = sol.minimumDeletions(nums)
    print(got, "OK" if got == want else "SAI")

5 OK
3 OK
1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (chỉ số lượng quan trọng).** Sau `a` lần xóa đầu và `b` lần xóa cuối (theo bất kỳ thứ tự nào), phần còn lại là `nums[a .. n − 1 − b]`. Do đó một chiến lược được mô tả đầy đủ bởi cặp `(a, b)` với `a + b ≤ n`, chi phí `a + b`.

**Bổ đề 2 (điều kiện hợp lệ).** Chỉ số `t` đã bị xóa ⇔ `t < a` hoặc `t ≥ n − b`.

**Ba phương án là đủ.** Với `i ≤ j`, một cặp `(a, b)` hợp lệ rơi vào một trong ba loại:
- `a > j`: cả hai bị xóa từ đầu ⇒ `a + b ≥ j + 1`, đạt được với `(j + 1, 0)`.
- `n − b ≤ i`: cả hai bị xóa từ cuối ⇒ `a + b ≥ n − i`, đạt được với `(0, n − i)`.
- còn lại: `i` phải bị xóa từ đầu (`a > i`, vì không bị xóa từ cuối) và `j` phải bị xóa từ cuối (`n − b ≤ j`) ⇒ `a + b ≥ (i + 1) + (n − j)`, đạt với `(i + 1, n − j)` (hợp lệ vì `i + 1 + n − j ≤ n` khi `i < j`; khi `i = j` phương án này lớn hơn hai phương án kia nên không ảnh hưởng `min`).

⇒ đáp án `= min` của ba giá trị. ✓

## 11. Phân tích Time / Space

`n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp `(a, b)` | `O(n²)` | `O(1)` |
| 2. Duyệt `a` | `O(n)` | `O(1)` |
| 3. Ba trường hợp | `O(n)` | `O(1)` |
| 4. BFS `(l, r)` | `O(n²)` | `O(n²)` |

In [13]:
rng = random.Random(1)
big = rng.sample(range(-10**5, 10**5 + 1), 10**5)
for name, fn in [("scan_a", deletions_scan_a), ("three_cases", deletions_three_cases), ("submit", Solution().minimumDeletions)]:
    t0 = time.perf_counter(); r = fn(big)
    print(f"{name:11s}: {r}  {(time.perf_counter() - t0) * 1000:.1f} ms")
t0 = time.perf_counter(); deletions_brute(big[:1500]); print(f"brute n = 1500: {time.perf_counter() - t0:.2f}s")

scan_a     : 46967  18.3 ms
three_cases: 46967  3.8 ms
submit     : 46967  3.7 ms


brute n = 1500: 0.08s


## 12. Bộ test

In [14]:
fns = [deletions_scan_a, deletions_three_cases, deletions_bfs, Solution().minimumDeletions]

for nums, want in examples:
    assert deletions_brute(nums) == want
    for fn in fns:
        assert fn(nums) == want

edge = [([1], 1), ([1, 2], 2), ([2, 1], 2), ([1, 2, 3], 2), ([3, 1, 2], 2), ([2, 3, 1], 2),
        ([1, 5, 2, 3, 4], 2),               # min, max ở hai vị trí đầu
        ([4, 2, 3, 1, 5], 2),               # min, max ở hai vị trí cuối
        ([1, 2, 3, 4, 5, 6, 7], 2),         # ở hai đầu mảng: mỗi bên 1
        ([3, 4, 1, 7, 2, 5, 6], 4)]         # min ở giữa: 1 tại 2, 7 tại 3 -> từ đầu 4
for nums, want in edge:
    assert deletions_brute(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

rng = random.Random(2024)
for _ in range(1500):
    n = rng.randint(1, 18)
    nums = rng.sample(range(-50, 51), n)
    want = deletions_brute(nums)
    for fn in fns:
        assert fn(nums) == want, (fn, nums)
print("ví dụ + edge + 1500 mảng ngẫu nhiên khớp brute force")

ví dụ + edge + 1500 mảng ngẫu nhiên khớp brute force


**Kiểm chứng toàn bộ:** vì chỉ thứ tự tương đối của các giá trị quan trọng, mọi mảng độ dài `n` tương đương với một **hoán vị** của `0..n−1`. Thử hết mọi hoán vị với `n ≤ 7` (`5913` mảng) là một bằng chứng mạnh hơn test ngẫu nhiên.

In [15]:
from itertools import permutations
total = 0
for n in range(1, 8):
    for perm in permutations(range(n)):
        nums = list(perm); want = deletions_brute(nums)
        assert deletions_three_cases(nums) == want and deletions_scan_a(nums) == want
        total += 1
print("đã kiểm tra mọi hoán vị độ dài 1..7:", total, "mảng")

đã kiểm tra mọi hoán vị độ dài 1..7: 5913 mảng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên phương án "mỗi bên một"

In [16]:
def wrong_two_options(nums):
    n = len(nums); i, j = sorted((nums.index(min(nums)), nums.index(max(nums))))
    return min(j + 1, n - i)                                   # SAI: thiếu (i + 1) + (n - j)

for nums, want in examples:
    print("sai =", wrong_two_options(nums), "| đúng =", want)

sai = 6 | đúng = 5
sai = 3 | đúng = 3
sai = 1 | đúng = 1


Ví dụ 1 cần đúng phương án hai bên (`2 + 3 = 5`). **Sửa:** thêm phương án (C).

### Lỗi 2 — Không sắp `i < j` trước khi tính

In [17]:
def wrong_no_sort(nums):
    n = len(nums); i, j = nums.index(min(nums)), nums.index(max(nums))   # SAI: có thể i > j
    return min(j + 1, n - i, i + 1 + n - j)

for nums, want in examples + [([5, 1, 9, 2], 2)]:
    print(f"{nums}: sai = {wrong_no_sort(nums)} | đúng = {want}")

[2, 10, 7, 5, 4, 1, 8, 6]: sai = 2 | đúng = 5
[0, -4, 19, 1, 8, -2, -3, 5]: sai = 3 | đúng = 3
[101]: sai = 1 | đúng = 1
[5, 1, 9, 2]: sai = 3 | đúng = 2


Với `i` = chỉ số min, `j` = chỉ số max mà `i > j`: "`j + 1`" chỉ xóa được max từ đầu (min vẫn còn), "`n − i`" chỉ xóa min từ cuối. Ví dụ 1: min ở 5, max ở 1 ⇒ `j + 1 = 2` — **nhỏ hơn đáp án thật** `5`. **Sửa:** `i, j = min(...), max(...)` của hai chỉ số.

### Lỗi 3 — Lệch một đơn vị khi đếm

In [18]:
def wrong_off_by_one(nums):
    n = len(nums); i, j = sorted((nums.index(min(nums)), nums.index(max(nums))))
    return min(j, n - i - 1, i + n - j)                        # SAI: quên phần tử mục tiêu cũng bị xóa

for nums, want in examples:
    print("sai =", wrong_off_by_one(nums), "| đúng =", want)

sai = 4 | đúng = 5
sai = 2 | đúng = 3
sai = 0 | đúng = 1


Xóa tới chỉ số `t` từ đầu tốn `t + 1` (gồm cả phần tử `t`), từ cuối tốn `n − t`. **Sửa:** cộng đúng. Ví dụ 3 (`[101]`) cho `0` là dấu hiệu rõ ràng.

### Lỗi 4 — Tin vào tham lam từng bước khi luật chi phí thay đổi

Tham lam "mỗi bước xóa ở đầu gần mục tiêu chưa xóa nhất" **tình cờ đúng** ở bài gốc: nếu `i ≤ n − 1 − j` thì nó xóa từ đầu tới `i` (`i + 1` bước) rồi chọn phía rẻ hơn cho `j`, cho `min(j + 1, i + 1 + n − j)`; phương án còn lại `n − i` không bao giờ nhỏ hơn vì `n − i ≥ j + 1 ⇔ i + j ≤ n − 1`. Nhưng chỉ cần đổi luật — ví dụ chi phí xóa đầu `cf` khác chi phí xóa cuối `cb` (biến thể 2) — tham lam cục bộ sẽ hỏng:

In [19]:
def greedy_step(nums, cf=1, cb=1):
    d = deque(range(len(nums)))
    tg = {nums.index(min(nums)), nums.index(max(nums))}; cost = 0
    while tg:
        front = min(k for k, t in enumerate(d) if t in tg)
        back = min(k for k, t in enumerate(reversed(d)) if t in tg)
        if (front + 1) * cf <= (back + 1) * cb:              # tối ưu cục bộ: mục tiêu gần nhất rẻ nhất
            t = d.popleft(); cost += cf
        else:
            t = d.pop(); cost += cb
        tg.discard(t)
    return cost

def weighted_opt(nums, cf, cb):
    n = len(nums); i, j = sorted((nums.index(min(nums)), nums.index(max(nums))))
    return min((j + 1) * cf, (n - i) * cb, (i + 1) * cf + (n - j) * cb)

rng = random.Random(4)
for _ in range(3000):
    nums = rng.sample(range(100), rng.randint(1, 9))
    assert greedy_step(nums) == deletions_brute(nums)         # bài gốc: tham lam đúng
print("bài gốc (cf = cb = 1): tham lam từng bước khớp brute force trên 3000 mảng")
nums, cf, cb = [56, 37, 17, 32, 48], 5, 2
print(f"cf = {cf}, cb = {cb}, nums = {nums}: tham lam = {greedy_step(nums, cf, cb)} | tối ưu = {weighted_opt(nums, cf, cb)}")

bài gốc (cf = cb = 1): tham lam từng bước khớp brute force trên 3000 mảng
cf = 5, cb = 2, nums = [56, 37, 17, 32, 48]: tham lam = 11 | tối ưu = 10


Ở phản ví dụ: min ở chỉ số 2, max ở 0. Tham lam xóa max từ đầu (`5`) rồi min ở giữa phải trả thêm `3 · 2 = 6` ⇒ `11`; tối ưu là xóa **tất cả từ cuối**: `5 · 2 = 10`. **Bài học:** lời giải nên dựa trên so sánh **toàn cục** các phương án (Cách 3); tham lam cục bộ cần chứng minh riêng cho từng luật.

### Lỗi 5 — Đếm hai lần khi min và max trùng nhau (`n = 1`)

Một số cách viết "cộng số lần xóa cho từng mục tiêu" sẽ ra `2` cho `[101]`. Công thức ba phương án tự xử lý (`min(1, 1, 2) = 1`); khi tự viết cách khác, luôn test `n = 1`.

In [20]:
def wrong_sum_each(nums):
    n = len(nums); i, j = nums.index(min(nums)), nums.index(max(nums))
    cost = lambda t: min(t + 1, n - t)
    return cost(i) + cost(j)                                    # SAI: tính riêng từng phần tử

for nums, want in examples + [([1, 2, 3, 4, 5, 6, 7], 2), ([3, 4, 1, 7, 2, 5, 6], 4)]:
    print(f"{nums}: sai = {wrong_sum_each(nums)} | đúng = {want}")

[2, 10, 7, 5, 4, 1, 8, 6]: sai = 5 | đúng = 5
[0, -4, 19, 1, 8, -2, -3, 5]: sai = 5 | đúng = 3
[101]: sai = 2 | đúng = 1
[1, 2, 3, 4, 5, 6, 7]: sai = 2 | đúng = 2
[3, 4, 1, 7, 2, 5, 6]: sai = 7 | đúng = 4


Cộng chi phí riêng từng phần tử vừa đếm trùng khi hai mục tiêu cùng phía (xóa phần tử xa đã kéo theo phần tử gần), vừa đếm 2 cho `n = 1`. **Sửa:** chi phí của **cả cặp** theo ba phương án.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ được xóa ở hai đầu" ⇒ phần còn lại là **đoạn liên tiếp**; mọi chiến lược = cặp `(a, b)`.
- Với một số ít mục tiêu: liệt kê các cách chia "mục tiêu nào bị lấy từ đầu, mục tiêu nào từ cuối" — luôn là **một tiền tố** các mục tiêu (theo vị trí) từ đầu và phần còn lại từ cuối.
- Sắp các chỉ số mục tiêu trước khi lập công thức.
- Test `n = 1`, mục tiêu ở hai đầu, mục tiêu kề nhau ở giữa.
- Họ bài: 1423 *Maximum Points You Can Obtain from Cards* (lấy hai đầu ⇔ cửa sổ ở giữa), 1658 *Minimum Operations to Reduce X to Zero*, 2516 *Take K of Each Character From Left and Right*.

## 15. Biến thể của bài

1. **Xóa một tập chỉ số bất kỳ** `T` (không chỉ min và max): sắp `T = t₁ < … < t_k`; đáp án `= min_{p=0..k} [ (t_p + 1 nếu p > 0) + (n − t_{p+1} nếu p < k) ]` (Bài 5).
2. **Chi phí khác nhau** cho xóa đầu (`cf`) và xóa cuối (`cb`): ba phương án nhân trọng số.
3. **Xóa `k` phần tử lớn nhất**: là trường hợp riêng của biến thể 1.
4. **Lấy lại thay vì xóa** (bài 1423): tối đa tổng lấy từ hai đầu với đúng `k` lần lấy ⇔ tối thiểu tổng cửa sổ ở giữa độ dài `n − k`.

In [21]:
def weighted_deletions(nums, cf, cb):
    """Biến thể 2: chi phí cf mỗi lần xóa đầu, cb mỗi lần xóa cuối."""
    n = len(nums); i, j = sorted((nums.index(min(nums)), nums.index(max(nums))))
    return min((j + 1) * cf, (n - i) * cb, (i + 1) * cf + (n - j) * cb)

def weighted_brute(nums, cf, cb):
    n = len(nums); tg = {nums.index(min(nums)), nums.index(max(nums))}; best = None
    for a in range(n + 1):
        for b in range(n - a + 1):
            if all(t < a or t >= n - b for t in tg):
                v = a * cf + b * cb; best = v if best is None else min(best, v)
    return best

rng = random.Random(15)
for _ in range(500):
    nums = rng.sample(range(60), rng.randint(1, 12)); cf, cb = rng.randint(1, 9), rng.randint(1, 9)
    assert weighted_deletions(nums, cf, cb) == weighted_brute(nums, cf, cb)
print("biến thể chi phí khác nhau khớp brute force; ví dụ 1 với cf = 1, cb = 5:",
      weighted_deletions([2, 10, 7, 5, 4, 1, 8, 6], 1, 5), "(xóa hết từ đầu: 6 rẻ hơn 2 + 15)")

biến thể chi phí khác nhau khớp brute force; ví dụ 1 với cf = 1, cb = 5: 6 (xóa hết từ đầu: 6 rẻ hơn 2 + 15)


**Biến thể 4 — lấy bài từ hai đầu (LeetCode 1423):** lấy đúng `k` lá từ đầu/cuối, tối đa tổng điểm. Phần **không** lấy là một đoạn liên tiếp độ dài `n − k` ⇒ đáp án = tổng toàn bộ − tổng nhỏ nhất của cửa sổ độ dài `n − k`. Cùng nhận xét "phần còn lại là đoạn liên tiếp" như bài này.

In [22]:
def max_score_cards(cards, k):
    n, w = len(cards), len(cards) - k
    if w == 0:
        return sum(cards)
    s = best = sum(cards[:w])
    for r in range(w, n):
        s += cards[r] - cards[r - w]; best = min(best, s)
    return sum(cards) - best

def max_score_cards_brute(cards, k):
    return max(sum(cards[:a]) + sum(cards[len(cards) - (k - a):]) for a in range(k + 1))

rng = random.Random(16)
for _ in range(500):
    cards = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, len(cards))
    assert max_score_cards(cards, k) == max_score_cards_brute(cards, k)
print("1423 ví dụ [1,2,3,4,5,6,1], k = 3 ->", max_score_cards([1, 2, 3, 4, 5, 6, 1], 3), "(kỳ vọng 12)")

1423 ví dụ [1,2,3,4,5,6,1], k = 3 -> 12 (kỳ vọng 12)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi `(a, b)` | định nghĩa | `O(n²)` | `O(1)` | kiểm chứng |
| 2. Duyệt `a` | `b` tối thiểu | `O(n)` | `O(1)` | nhiều mục tiêu |
| 3. Ba trường hợp | công thức đóng | `O(n)` | `O(1)` | **nộp bài** |
| 4. BFS `(l, r)` | mô hình trạng thái | `O(n²)` | `O(n²)` | trọng tài độc lập |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[5, 3, 9, 1, 7]`? `[1, 9]`? `[4, 9, 1, 3, 2, 6, 5, 8, 7]`?

**Bài 2 (tính tay):** `nums = [6, 2, 8, 4, 10, 9, 1, 3]`. Tìm `i`, `j`, ba phương án và đáp án.

**Bài 3 (điền TODO):** hoàn thiện `min_max_indices(nums)` trả về `(chỉ số min, chỉ số max)` trong **một** lượt duyệt.

**Bài 4 (tự viết lại):** viết `deletion_plan(nums)` trả về cặp `(a, b)` — số lần xóa đầu và cuối — của một phương án tối ưu; kiểm tra `a + b` bằng đáp án và min/max đều bị xóa.

**Bài 5 (mini challenge):** cho `nums` và tập chỉ số `targets` (có thể nhiều phần tử). Tìm số thao tác xóa hai đầu ít nhất để mọi chỉ số trong `targets` đều bị xóa, trong `O(k log k)` sau khi sắp. Kiểm chứng bằng brute force `(a, b)`. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [23]:
def min_max_indices(nums):
    # TODO: giữ i (min) và j (max), cập nhật khi gặp phần tử nhỏ hơn / lớn hơn
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Tìm vị trí min/max. (2) `[5,3,9,1,7]`: max ở 2, min ở 3 ⇒ A = 4, B = 3, C = 3 + 2 = 5. (3) Đáp án `3`; `[1, 9]` ⇒ `2`; mảng thứ ba: min ở 2, max ở 1 ⇒ `i = 1, j = 2`, `n = 9`: A = 3.

**Bài 2:** (1) Min = 1 ở 6, max = 10 ở 4. (2) `i = 4`, `j = 6`, `n = 8`. (3) A = 7, B = 4, C = 5 + 2 = 7 ⇒ `4`.

**Bài 3:** (1) `i = j = 0`. (2) So sánh từng phần tử với `nums[i]`, `nums[j]`. (3) Trả `(i, j)`.

**Bài 4:** (1) Tính ba phương án như Cách 3. (2) Ánh xạ: A ↦ `(j+1, 0)`, B ↦ `(0, n−i)`, C ↦ `(i+1, n−j)`. (3) Kiểm tra bằng điều kiện `t < a or t ≥ n − b`.

**Bài 5:** (1) Sắp `targets` tăng dần: `t₁ < … < t_k`. (2) Phương án `p`: `p` mục tiêu đầu xóa từ đầu (`t_p + 1`), còn lại từ cuối (`n − t_{p+1}`). (3) Lấy `min` trên `p = 0..k`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
for nums, want in [([5, 3, 9, 1, 7], 3), ([1, 9], 2), ([4, 9, 1, 3, 2, 6, 5, 8, 7], 3)]:
    assert Solution().minimumDeletions(nums) == deletions_brute(nums) == want
# Bài 2
assert trace_three([6, 2, 8, 4, 10, 9, 1, 3]) == 4

# Bài 3
def min_max_indices_ans(nums):
    i = j = 0
    for t, x in enumerate(nums):
        if x < nums[i]: i = t
        if x > nums[j]: j = t
    return i, j

# Bài 4
def deletion_plan_ans(nums):
    n = len(nums); i, j = sorted(min_max_indices_ans(nums))
    plans = [(j + 1, 0), (0, n - i), (i + 1, n - j)]
    return min(plans, key=lambda ab: ab[0] + ab[1])

rng = random.Random(41)
for _ in range(1000):
    nums = rng.sample(range(-30, 31), rng.randint(1, 15)); n = len(nums)
    assert min_max_indices_ans(nums) == (nums.index(min(nums)), nums.index(max(nums)))
    a, b = deletion_plan_ans(nums)
    assert a + b == deletions_brute(nums) and a + b <= n
    assert all(t < a or t >= n - b for t in min_max_indices_ans(nums))
print("Bài 4 ví dụ 1:", deletion_plan_ans([2, 10, 7, 5, 4, 1, 8, 6]))
print("Bài 1-4: OK")

nums = [6, 2, 8, 4, 10, 9, 1, 3]
  min = 1 tại 6, max = 10 tại 4
  i = 4, j = 6, n = 8: A = 7, B = 4, C = 5 + 2 = 7 -> 4 (B: xóa từ cuối)
Bài 4 ví dụ 1: (2, 3)
Bài 1-4: OK


In [25]:
# Bài 5 — xóa một tập chỉ số bất kỳ
def remove_targets(n, targets):
    t = sorted(set(targets)); k = len(t)
    if k == 0:
        return 0
    best = n
    for p in range(k + 1):                      # p mục tiêu đầu lấy từ đầu
        front = t[p - 1] + 1 if p > 0 else 0
        back = n - t[p] if p < k else 0
        best = min(best, front + back)
    return best

def remove_targets_brute(n, targets):
    best = n
    for a in range(n + 1):
        for b in range(n - a + 1):
            if all(x < a or x >= n - b for x in targets):
                best = min(best, a + b)
    return best

rng = random.Random(55)
for _ in range(1000):
    n = rng.randint(1, 14)
    tg = rng.sample(range(n), rng.randint(0, n))
    assert remove_targets(n, tg) == remove_targets_brute(n, tg), (n, tg)
nums = [2, 10, 7, 5, 4, 1, 8, 6]
top3 = sorted(range(len(nums)), key=lambda i: -nums[i])[:3]
print("Bài 5: xóa 3 phần tử lớn nhất của ví dụ 1 (chỉ số", sorted(top3), ") cần", remove_targets(len(nums), top3), "thao tác")
assert remove_targets(len(nums), [nums.index(min(nums)), nums.index(max(nums))]) == 5

Bài 5: xóa 3 phần tử lớn nhất của ví dụ 1 (chỉ số [1, 2, 6] ) cần 5 thao tác


## 20. Template nhớ nhanh + Checklist

```python
# Xóa ở hai đầu để loại hai chỉ số i <= j
i, j = sorted((idx_min, idx_max))
ans = min(j + 1,             # tất cả từ đầu
          n - i,             # tất cả từ cuối
          i + 1 + n - j)     # i từ đầu, j từ cuối
# Tổng quát (nhiều mục tiêu t1 < ... < tk): min_p (t_p + 1) + (n - t_{p+1})
```

**Checklist trước khi submit:**
- [ ] Đủ **ba** phương án.
- [ ] Sắp hai chỉ số `i ≤ j` trước khi áp công thức.
- [ ] Xóa tới chỉ số `t`: từ đầu `t + 1`, từ cuối `n − t`.
- [ ] `n = 1` cho `1` (min trùng max).
- [ ] So sánh toàn cục ba phương án (tham lam cục bộ chỉ đúng tình cờ với chi phí bằng nhau).